* lr = 3e - 4
* batch size = 61
* weights = ResNet18_Weights.IMAGENET1K_V1
* model = resnet18(weights=weights)
* Norm : mean = IMAGENET_MEAN, std = IMAGENET_STD

# 01 ) 설치 & 기본 임포트

In [ ]:
!pip -q install ultralytics matplotlib opencv-python pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.9 MB/s eta 0:00:00


In [ ]:
import os, json, math, glob
from pathlib import Path
from typing import List, Dict, Any, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from tqdm.auto import tqdm
from IPython.display import display

from google.colab import drive

from multiprocessing import cpu_count

# 02 ) 환경 / 경로 설정

In [ ]:
drive.mount('/content/drive')

CFG = {
    # 루트 경로
    "DATA_ROOT": "/content/drive/MyDrive/Data_Creater_Camp",

    # 원천/라벨 경로
    "TRAIN_SRC": "Training/01.원천데이터/TS_KS",
    "TRAIN_LBL": "Training/02.라벨링데이터/TL_KS_LINE",
    "VALID_SRC": "Validation/01.원천데이터/VS_KS",
    "VALID_LBL": "Validation/02.라벨링데이터/VL_KS_LINE",

    # 전처리 산출물(크롭 X, index.csv & labels만)
    "DATASET_DIR": "ResNet_Dataset",

    # 러닝 파라미터
    "BATCH_SIZE": 64,
    # Set num_workers to 0 to debug DataLoader worker issues.
    # If this resolves the error, you can try increasing it gradually.
    "NUM_WORKERS": 2, #min(8, cpu_count()),
    "LR": 3e-4,     #0.0003
    "MAX_EPOCHS": 80,
    "PATIENCE": 10,
    "BEST_CKPT_NAME": "resnet18_lineheight_best.pt",

    # Transform 통계
    "IMAGENET_MEAN": [0.485, 0.456, 0.406],
    "IMAGENET_STD":  [0.229, 0.224, 0.225],
}

# 파생 경로 구성
DATA_ROOT = Path(CFG["DATA_ROOT"])
Train_Source_DIR       = str(DATA_ROOT / CFG["TRAIN_SRC"])
Train_Label_DIR        = str(DATA_ROOT / CFG["TRAIN_LBL"])
Validation_Source_DIR  = str(DATA_ROOT / CFG["VALID_SRC"])
Validation_Label_DIR   = str(DATA_ROOT / CFG["VALID_LBL"])

DATASET_DIR = DATA_ROOT / CFG["DATASET_DIR"]
for sub in ["train/images", "train/labels", "valid/images", "valid/labels"]:
    (DATASET_DIR / sub).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

Mounted at /content/drive
DEVICE: cuda


# 03 ) 공통 유틸

In [ ]:
def _safe_get(d: Any, key: str, default=None):
    return d[key] if (isinstance(d, dict) and key in d) else default

def _coalesce(*vals, default=None):
    for v in vals:
        if v is not None:
            return v
    return default

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# 03 ) JSON -> 라인 아이템 파싱

In [ ]:
def load_line_labels_recursive(label_dir: str, desc="parse") -> List[Dict[str, Any]]:
    items = []
    json_list = sorted(Path(label_dir).rglob("*.json"))
    for jf in tqdm(json_list, desc=f"[{desc}] JSON files", unit="file"):
        try:
            with open(jf, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"JSON 읽기 실패: {jf} -> {e}")
            continue

        entries = list(data.values()) if isinstance(data, dict) else (data if isinstance(data, list) else [])

        for entry in entries:
            filename = _safe_get(entry, "filename", None)
            regions  = _safe_get(entry, "regions", [])
            if not filename or not regions:
                continue

            for region in regions:
                shape = _safe_get(region, "shape_attributes", {})
                attrs = _safe_get(region, "region_attributes", {})
                name  = str(_safe_get(shape, "name", "")).lower()

                # 좌표 추출 (line 혹은 polyline의 양 끝점)
                x1 = y1 = x2 = y2 = None
                if name == "line":
                    x1 = float(_safe_get(shape, "x1", float("nan")))
                    y1 = float(_safe_get(shape, "y1", float("nan")))
                    x2 = float(_safe_get(shape, "x2", float("nan")))
                    y2 = float(_safe_get(shape, "y2", float("nan")))

                elif name == "polyline":
                    # 키 호환: all_point_x / all_points_x (VIA 버전차)
                    xs = _coalesce(_safe_get(shape, "all_points_x", None),
                                   _safe_get(shape, "all_point_x", None),
                                   default=[])
                    ys = _coalesce(_safe_get(shape, "all_points_y", None),
                                   _safe_get(shape, "all_point_y", None),
                                   default=[])
                    if isinstance(xs, list) and isinstance(ys, list) and len(xs) >= 2 and len(ys) >= 2:
                        x1, y1 = float(xs[0]),  float(ys[0])
                        x2, y2 = float(xs[-1]), float(ys[-1])

                else:
                    continue

                if any(map(lambda v: v is None or (isinstance(v, float) and pd.isna(v)), [x1,y1,x2,y2])):
                    continue

                # height 키 호환
                height_raw = _coalesce(_safe_get(attrs, "height", None),
                                       _safe_get(attrs, "chi_height_m", None),
                                       _safe_get(attrs, "h", None))
                try:
                    height = float(height_raw)
                except:
                    continue

                items.append({
                    "img_path": filename,
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "height": height,
                    "chi_id": _safe_get(attrs, "chi_id", None)
                })
    return items

# 05 ) index.csv + height 라벨 텍스트 내보내기

In [ ]:
# 하나의 이미지에 다중 라인 -> _1, _2 와 같이 번호 부여
# 경로 미스매치 시 후보 탐색

def export_index_and_labels_only(
    items: List[Dict[str, Any]],
    image_dir: str,
    out_label_dir: str,
    out_index_csv: str,
    data_root: str,
    desc="index"
) -> pd.DataFrame:
    out_label_dir = Path(out_label_dir)
    out_label_dir.mkdir(parents=True, exist_ok=True)

    rows, per_image_counter = [], {}
    warn_missing = 0
    for it in tqdm(items, desc=f"[{desc}] build rows", unit="line"):
        img_file = Path(image_dir) / it["img_path"]
        if not img_file.exists():
            # 1) 같은 파일명 재귀검색 (image_dir 아래)
            cands = list(Path(image_dir).rglob(Path(it["img_path"]).name))
            if cands:
                img_file = cands[0]
            else:
                # 2) DATA_ROOT 전역 재귀검색
                cands2 = list(Path(data_root).rglob(Path(it["img_path"]).name))
                if cands2:
                    img_file = cands2[0]
                else:
                    warn_missing += 1
                    if warn_missing <= 10:
                        print(f"이미지를 찾을 수 없습니다.: {it['img_path']}")
                    continue

        stem = img_file.stem
        per_image_counter[stem] = per_image_counter.get(stem, 0) + 1
        cnt = per_image_counter[stem]
        label_filename = f"{stem}_{cnt}.txt"

        # height만 저장
        (out_label_dir / label_filename).write_text(str(it["height"]), encoding="utf-8")

        # DATA_ROOT 기준의 상대경로 저장 (이식성 ↑)
        data_root_path = Path(data_root)
        try:
            rel_img_path = str(img_file.relative_to(data_root_path))
        except Exception:
            rel_img_path = str(img_file)

        rows.append({
            "img_path": rel_img_path,
            "label_file": label_filename,
            "chi_id": it["chi_id"],
            "height": it["height"],
            "x1": it["x1"], "y1": it["y1"], "x2": it["x2"], "y2": it["y2"],
        })

    if warn_missing > 10:
        print(f"이미지를 찾을 수 없습니다 : {warn_missing - 10} 개 이상")

    df = pd.DataFrame(rows, columns=["img_path","label_file","chi_id","height","x1","y1","x2","y2"])
    Path(out_index_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_index_csv, index=False, encoding="utf-8")
    print(f"Saved index csv: {out_index_csv} ({len(df)} rows)")
    return df

# 06 ) 패치 추출 & Dataset

In [ ]:
def crop_line_patch_np(img_bgr: np.ndarray, x1, y1, x2, y2, out_size=224) -> np.ndarray:
    H, W = img_bgr.shape[:2]
    cx, cy = (x1 + x2) * 0.5, (y1 + y2) * 0.5
    angle_rad = math.atan2((y2 - y1), (x2 - x1))
    angle_deg = np.degrees(angle_rad)

    M = cv2.getRotationMatrix2D((cx, cy), -angle_deg, 1.0)
    rotated = cv2.warpAffine(img_bgr, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

    rcx, rcy = (M @ np.array([cx, cy, 1.0], dtype=np.float32)).tolist()
    half = out_size // 2
    x_min, y_min = int(round(rcx - half)), int(round(rcy - half))
    x_max, y_max = x_min + out_size, y_min + out_size

    pad_left   = max(0, -x_min)
    pad_top    = max(0, -y_min)
    pad_right  = max(0, x_max - rotated.shape[1])
    pad_bottom = max(0, y_max - rotated.shape[0])
    if any([pad_left, pad_top, pad_right, pad_bottom]):
        rotated = cv2.copyMakeBorder(rotated, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_REFLECT)
        x_min += pad_left; x_max += pad_left
        y_min += pad_top;  y_max += pad_top

    patch = rotated[y_min:y_max, x_min:x_max]
    if patch.shape[:2] != (out_size, out_size):
        patch = cv2.resize(patch, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)

class IndexCSVHeightDataset(Dataset):
    def __init__(self, index_csv: str, data_root: str, transform=None, use_shape_geom: bool = True):
        self.df = pd.read_csv(index_csv)
        self.data_root = Path(data_root)
        self.transform = transform
        self.use_shape_geom = use_shape_geom

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path_abs = self.data_root / row["img_path"]
        img_bgr = cv2.imread(str(img_path_abs), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f"이미지를 찾을 수 없습니다 : {img_path_abs}")

        if self.use_shape_geom:
            # 학습 전용: 라벨 기반 회전 + 크롭 허용
            patch_rgb = crop_line_patch_np(
                img_bgr, row["x1"], row["y1"], row["x2"], row["y2"], out_size=224
            )
            pil_img = Image.fromarray(patch_rgb)
        else:
            # 검증 전용: 라벨 미사용, 전체 이미지를 center-square 후 리사이즈(224)
            h, w = img_bgr.shape[:2]
            side = min(h, w)
            y0 = (h - side) // 2
            x0 = (w - side) // 2
            square = img_bgr[y0:y0+side, x0:x0+side]
            square = cv2.cvtColor(square, cv2.COLOR_BGR2RGB)
            square = cv2.resize(square, (224, 224), interpolation=cv2.INTER_LINEAR)
            pil_img = Image.fromarray(square)


        if self.transform is not None:
            pil_img = self.transform(pil_img)

        target = torch.tensor([float(row["height"])], dtype=torch.float32)
        return pil_img, target

# 07 ) Transform

In [ ]:
IMAGENET_MEAN = CFG["IMAGENET_MEAN"]
IMAGENET_STD  = CFG["IMAGENET_STD"]

train_transform = transforms.Compose([
    # 이미지 회전
    transforms.RandomRotation(degrees=15),
    # 색상, 대비, 채도, 색조 변경
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    # 좌우 반전
    transforms.RandomHorizontalFlip(p=0.5),
    # 상하 반전
    transforms.RandomVerticalFlip(p=0.5),
    # PyTorch Tensor로 변환
    transforms.ToTensor(),
    # 이미지넷 평균/표준편차로 정규화
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

valid_transform = transforms.Compose([
    # PyTorch Tensor로 변환
    transforms.ToTensor(),
    # 이미지넷 평균/표준편차로 정규화
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# 08 ) 인덱스 생성 (train / valid)

In [ ]:
# 08 ) 인덱스 준비: 있으면 "읽기만", 없으면 생성
from pathlib import Path

train_index_csv = DATASET_DIR / "train" / "index.csv"
valid_index_csv = DATASET_DIR / "valid" / "index.csv"

REBUILD_INDEX = False  # 필요 시 True로 강제 재생성

def _index_ok(p: Path, required=("img_path","label_file","chi_id","height","x1","y1","x2","y2")) -> bool:
    if not p.exists() or p.stat().st_size == 0:
        return False
    try:
        cols = set(pd.read_csv(p, nrows=0).columns)
    except Exception:
        return False
    return set(required).issubset(cols)

if (not REBUILD_INDEX) and _index_ok(train_index_csv) and _index_ok(valid_index_csv):
    print("[index.csv] 가 존재합니다.")
    print("\n[train/index.csv head]"); display(pd.read_csv(train_index_csv).head())
    print("\n[valid/index.csv head]"); display(pd.read_csv(valid_index_csv).head())
else:
    print("[index.csv] 가 존재하지 않습니다. → index.csv를 생성합니다. \n")

    print("DATA_ROOT :", DATA_ROOT)
    print("TRAIN IMG :", Train_Source_DIR)
    print("TRAIN JSON:", Train_Label_DIR)
    print("VALID IMG :", Validation_Source_DIR)
    print("VALID JSON:", Validation_Label_DIR)
    print("DATASET  :", DATASET_DIR)

    for p in [Train_Source_DIR, Train_Label_DIR, Validation_Source_DIR, Validation_Label_DIR]:
        print(f"[exists] {p} ->", os.path.exists(p))

    train_items = load_line_labels_recursive(Train_Label_DIR,  desc="train-parse")
    valid_items = load_line_labels_recursive(Validation_Label_DIR, desc="valid-parse")
    print(f"Parsed items -> train: {len(train_items)}, valid: {len(valid_items)}")

    _ = export_index_and_labels_only(
        train_items, Train_Source_DIR, DATASET_DIR / "train" / "labels",
        train_index_csv, DATA_ROOT, desc="train-index"
    )
    _ = export_index_and_labels_only(
        valid_items, Validation_Source_DIR, DATASET_DIR / "valid" / "labels",
        valid_index_csv, DATA_ROOT, desc="valid-index"
    )

    if train_index_csv.exists():
        print("\n[train/index.csv head]")
        display(pd.read_csv(train_index_csv).head())
    if valid_index_csv.exists():
        print("\n[valid/index.csv head]")
        display(pd.read_csv(valid_index_csv).head())


[index.csv] 가 존재합니다.

[train/index.csv head]


,img_path,label_file,chi_id,height,x1,y1,x2,y2
0,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_0_1.txt,1,76.78,108.0,378.0,184.0,370.0
1,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_0_2.txt,2,63.81,221.0,402.0,284.0,394.0
2,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_1_1.txt,1,76.78,109.0,122.0,185.0,114.0
3,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_1_2.txt,2,64.81,221.0,146.0,285.0,138.0
4,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_10_1.txt,1,105.08,100.0,236.0,204.0,225.0



[valid/index.csv head]


,img_path,label_file,chi_id,height,x1,y1,x2,y2
0,Validation/01.원천데이터/VS_KS/K3A_CHN_201611120524...,K3A_CHN_20161112052404_15_1.txt,1,137.59,132.0,412.0,268.0,396.0
1,Validation/01.원천데이터/VS_KS/K3A_CHN_201701150511...,K3A_CHN_20170115051130_1_1.txt,1,108.90,389.0,112.0,457.0,101.0
2,Validation/01.원천데이터/VS_KS/K3A_CHN_201701230521...,K3A_CHN_20170123052151_1_1.txt,1,107.28,328.0,137.0,382.0,128.0
3,Validation/01.원천데이터/VS_KS/K3A_CHN_201701230521...,K3A_CHN_20170123052151_14_1.txt,1,99.56,444.0,132.0,494.0,123.0
4,Validation/01.원천데이터/VS_KS/K3A_CHN_201701230521...,K3A_CHN_20170123052151_22_1.txt,1,113.09,140.0,173.0,197.0,164.0


# 09 ) DataLoader

In [ ]:
train_ds = IndexCSVHeightDataset(str(train_index_csv), str(DATA_ROOT), transform=train_transform, use_shape_geom=True)
valid_ds = IndexCSVHeightDataset(str(valid_index_csv), str(DATA_ROOT), transform=valid_transform, use_shape_geom=True)

# 안전한 pin_memory_device 설정 및 DataLoader 생성 (교체해서 사용)
pin_mem_dev = "cuda" if DEVICE.type == "cuda" else "cpu"

train_loader_kwargs = dict(
    dataset=train_ds,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=True,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=True,
    persistent_workers=True, # Set to False when num_workers is 0
    prefetch_factor=None, # Set to None when num_workers is 0
    drop_last=True,
)

valid_loader_kwargs = dict(
    dataset=valid_ds,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=True,
    persistent_workers=True, # Set to False when num_workers is 0
    prefetch_factor=None, # Set to None when num_workers is 0
    drop_last=False,
)

# 1) 우선 pin_memory_device를 넣어 시도 (PyTorch 버전에 따라 다름)
try:
    train_loader = DataLoader(**train_loader_kwargs, pin_memory_device=pin_mem_dev)
    valid_loader = DataLoader(**valid_loader_kwargs, pin_memory_device=pin_mem_dev)
    print(f"[DataLoader] using pin_memory_device='{pin_mem_dev}'")
except TypeError:
    # 예: 일부 PyTorch 버전 / 환경에서는 pin_memory_device 인자를 지원하지 않거나
    # 내부 구현이 다른 타입을 기대할 수 있음 → 안전하게 pin_memory만 사용
    train_loader = DataLoader(**train_loader_kwargs)
    valid_loader = DataLoader(**valid_loader_kwargs)
    print("[DataLoader] pin_memory_device not supported — fallback to pin_memory only")

[DataLoader] using pin_memory_device='cuda'


# 10 ) 모델 / 학습 유틸

In [ ]:
def build_model() -> nn.Module:
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model.to(DEVICE)

criterion = nn.MSELoss()

def rmse_from_mse(mse_val: float) -> float:
    return float(np.sqrt(mse_val))

def train_one_epoch(model, loader, optimizer, epoch=None):
    model.train()
    total_loss, n = 0.0, 0
    iterator = tqdm(loader, desc=f"[train] epoch {epoch}" if epoch else "[train]", unit="batch", leave=False)

    scaler = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    for images, targets in iterator:
        images  = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', enabled=(DEVICE.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        n += images.size(0)
        iterator.set_postfix(rmse=np.sqrt(total_loss / max(1, n)))
    return float(np.sqrt(total_loss / max(1, n)))

@torch.no_grad()
def evaluate(model, loader, desc="[valid]"):
    model.eval()
    preds_list, gts_list = [], []
    total_loss, n = 0.0, 0

    for images, targets in tqdm(loader, desc=desc, unit="batch", leave=False):
        images  = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', enabled=(DEVICE.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, targets)

        total_loss += loss.item() * images.size(0)
        n += images.size(0)
        preds_list.append(outputs.detach().cpu().numpy())
        gts_list.append(targets.detach().cpu().numpy())

    mse = total_loss / max(1, n)
    rmse = float(np.sqrt(mse))
    preds = np.concatenate(preds_list, axis=0).reshape(-1)
    gts   = np.concatenate(gts_list,   axis=0).reshape(-1)
    return rmse, mse, preds, gts

class EarlyStopping:
    def __init__(self, patience=10, mode='min'):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.num_bad = 0
        self.is_better = (lambda a, b: a < b) if mode == 'min' else (lambda a, b: a > b)

    def step(self, metric):
        if self.best is None or self.is_better(metric, self.best):
            self.best = metric
            self.num_bad = 0
            return True
        self.num_bad += 1
        return False

def rmse_image_level(preds: np.ndarray, gts: np.ndarray, index_csv_path: str) -> float:
    """
    index.csv의 img_path 단위로 평균을 취해 이미지 레벨 RMSE 계산.
    (DataLoader는 valid에서 shuffle=False 이므로 순서 정합)
    """
    df = pd.read_csv(index_csv_path)
    assert len(df) == len(preds) == len(gts), "index.csv 행 수와 preds/gts 길이가 일치해야 합니다."
    df_eval = pd.DataFrame({"img_path": df["img_path"].values, "pred": preds, "gt": gts})
    grouped = df_eval.groupby("img_path").agg({"pred":"mean","gt":"mean"}).reset_index()
    return float(np.sqrt(np.mean((grouped["pred"].values - grouped["gt"].values) ** 2)))

# 11 ) 학습 루프

In [ ]:
model = build_model()
optimizer = optim.Adam(model.parameters(), lr=CFG["LR"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["MAX_EPOCHS"], eta_min=1e-6
)

early = EarlyStopping(patience=CFG["PATIENCE"], mode='min')
BEST_CKPT = str(DATASET_DIR / CFG["BEST_CKPT_NAME"])
best_rmse = float("inf")



for epoch in range(1, CFG["MAX_EPOCHS"] + 1):
    tr_rmse = train_one_epoch(model, train_loader, optimizer, epoch=epoch)
    va_rmse, va_mse, _, _ = evaluate(model, valid_loader, desc=f"[valid] epoch {epoch}")
    scheduler.step()

    msg = f"[Epoch {epoch:03d}] train RMSE: {tr_rmse:.4f} | valid RMSE: {va_rmse:.4f}"
    try:
        # prefer tqdm.write if available and working
        from tqdm import std as _tqdm_std   # safe import for attribute
        # attempt use tqdm.write from whichever tqdm variant is available
        tqdm.write(msg)
    except Exception:
        # fallback: plain print (guaranteed)
        print(msg)

    if early.step(va_rmse):
        best_rmse = va_rmse
        torch.save(model.state_dict(), BEST_CKPT)
        tqdm.write(f"  ↳ New best! Saved checkpoint to: {BEST_CKPT}")
    elif early.num_bad >= CFG["PATIENCE"]:
        tqdm.write(f"Early stopping at epoch {epoch}. Best valid RMSE: {best_rmse:.4f}")
        break

[train] epoch 1:   0%|          | 0/165 [00:00<?, ?batch/s]

[valid] epoch 1:   0%|          | 0/21 [00:00<?, ?batch/s]

[Epoch 001] train RMSE: 103.1934 | valid RMSE: 84.5029
  ↳ New best! Saved checkpoint to: /content/drive/MyDrive/Data_Creater_Camp/ResNet_Dataset/resnet18_lineheight_best.pt


[train] epoch 2:   0%|          | 0/165 [00:00<?, ?batch/s]

[valid] epoch 2:   0%|          | 0/21 [00:00<?, ?batch/s]

...

[valid] epoch 79:   0%|          | 0/21 [00:00<?, ?batch/s]

[Epoch 079] train RMSE: 3.4362 | valid RMSE: 6.2348


[train] epoch 80:   0%|          | 0/165 [00:00<?, ?batch/s]

[valid] epoch 80:   0%|          | 0/21 [00:00<?, ?batch/s]

[Epoch 080] train RMSE: 3.1524 | valid RMSE: 6.2611


# 12 ) 최종 평가

In [ ]:
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
model.to(DEVICE)

final_rmse, final_mse, preds, gts = evaluate(model, valid_loader, desc="[valid] final")
print(f"\n[Final] Line-level RMSE: {final_rmse:.6f}")

img_level_rmse = rmse_image_level(preds, gts, str(valid_index_csv))
print(f"[Final] Image-level RMSE: {img_level_rmse:.6f}")

[valid] final:   0%|          | 0/21 [00:00<?, ?batch/s]


[Final] Line-level RMSE: 6.220362
[Final] Image-level RMSE: 5.982751


1. DRY 원칙 : 설정과 함수를 한 곳에 모아 하이퍼파라미터 수정 등을 할때, 한 군데만 수정 하면 됨 -> 유지보수성 향상

2. 스키마 불변성 확보 : 라벨 파서는 다양한 JSON 변이를 단일 함수로 수용, 다른 종류의 데이터가 소스가 들어와도 파이프라인이 깨지지 않음 -> 안정성 강화

3. 재현 가능성 : 시드를 고정하여 검증 간 분산 감소

1. 주어진 데이터에서 file_attributes값은 사용 불가합니다. (사진 해상도, 각도)
=> *통과*

2. train, validation 두 데이터셋에서 shape_attribute 값 (굴뚝 위치)은 제공됩니다.

3. validation 셋에서는 기본적인 augmentation (회전, 리사이즈 등) 외의
“shape_attribution을 활용한“ 이미지 변형(crop 등) 은 불가능합니다.
=> *통과*